Data validation and schema enforcement
Goal: Validate schema, enforce data types, and handle missing values

In [1]:
# Import libraries and load data
print("="*70)
print("Step 2: Data validation and schema enforcement")
print("="*70)

import pandas as pd
import numpy as np
from pathlib import Path

# File paths
TRAIN_FILE = Path("eurlex.csv")
TEST_FILE = Path("eurlex_test.csv")

# Load datasets
print("[1/5] Loading datasets...")
df_train = pd.read_csv(TRAIN_FILE)
df_test = pd.read_csv(TEST_FILE)

print(f"[1/5] Train shape: {df_train.shape}")
print(f"[1/5] Test shape: {df_test.shape}")

Step 2: Data validation and schema enforcement
[1/5] Loading datasets...
[1/5] Train shape: (15377, 5002)
[1/5] Test shape: (3971, 5001)


In [2]:
# Schema validation
print("[2/5] Validating schema...")

# Check column names
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

# Test should have all columns except 'labels'
expected_test_cols = train_cols - {'labels'}

if test_cols == expected_test_cols:
    print("[2/5] Test columns match expected schema")
else:
    print("[2/5] Schema mismatch detected!")
    print(f"  Missing in test: {expected_test_cols - test_cols}")
    print(f"  Extra in test: {test_cols - expected_test_cols}")

# Verify feature columns
feature_cols = [col for col in df_train.columns if col.startswith('f')]
print(f"[2/5] Number of feature columns: {len(feature_cols)}")

[2/5] Validating schema...
[2/5] Test columns match expected schema
[2/5] Number of feature columns: 5000


In [3]:
# Data type validation and enforcement
print("[3/5] Validating and enforcing data types...")

# row_id should be integer
df_train['row_id'] = df_train['row_id'].astype(int)
df_test['row_id'] = df_test['row_id'].astype(int)
print("[3/5] row_id converted to int")

# All feature columns should be numeric (float)
for col in feature_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce')

print(f"[3/5] All {len(feature_cols)} feature columns converted to numeric")

# labels column should be string
df_train['labels'] = df_train['labels'].astype(str)
print("[3/5] labels column set as string")

print("[3/5] Data types summary:")
print(f"  row_id: {df_train['row_id'].dtype}")
print(f"  features: {df_train[feature_cols[0]].dtype}")
print(f"  labels: {df_train['labels'].dtype}")

[3/5] Validating and enforcing data types...
[3/5] row_id converted to int
[3/5] All 5000 feature columns converted to numeric
[3/5] labels column set as string
[3/5] Data types summary:
  row_id: int64
  features: float64
  labels: object


In [4]:
# Missing value analysis and handling
print("[4/5] Analyzing missing values...")

# Check for missing values in features
train_missing = df_train[feature_cols].isnull().sum().sum()
test_missing = df_test[feature_cols].isnull().sum().sum()

print(f"[4/5] Train missing values in features: {train_missing}")
print(f"[4/5] Test missing values in features: {test_missing}")

if train_missing > 0 or test_missing > 0:
    print("[4/5] Missing values detected! Handling strategy:")
    print("  -> Filling missing values with 0.0 (appropriate for sparse features)")
    # Fill missing values with 0.0
    df_train[feature_cols] = df_train[feature_cols].fillna(0.0)
    df_test[feature_cols] = df_test[feature_cols].fillna(0.0)
    # Verify
    train_missing_after = df_train[feature_cols].isnull().sum().sum()
    test_missing_after = df_test[feature_cols].isnull().sum().sum()
    print(f"[4/5] After filling - Train missing: {train_missing_after}, Test missing: {test_missing_after}")
else:
    print("[4/5] No missing values detected")

# Check for missing labels in train
train_label_missing = df_train['labels'].isnull().sum()
print(f"[4/5] Missing labels in train: {train_label_missing}")

if train_label_missing > 0:
    print("[4/5] Warning: Some training samples have no labels")
    print("  -> These will be assigned empty label set ''")
    df_train['labels'] = df_train['labels'].fillna('')

[4/5] Analyzing missing values...
[4/5] Train missing values in features: 0
[4/5] Test missing values in features: 0
[4/5] No missing values detected
[4/5] Missing labels in train: 0


In [5]:
# Data quality summary report
print("[5/5] Data quality summary report")
print("="*70)

def get_stats(df, name):
    
    print(f"\n{name} dataset:")
    print(f"  Rows: {len(df):,}")
    
    print(f"  Features: {len(feature_cols)}")
    # Feature statistics
    feature_data = df[feature_cols]
    print(f"  Feature value range: [{feature_data.min().min():.4f}, {feature_data.max().max():.4f}]")
    print(f"  Feature mean: {feature_data.mean().mean():.4f}")
    print(f"  Feature std: {feature_data.std().mean():.4f}")
    # Sparsity
    zero_count = (feature_data == 0).sum().sum()
    total_elements = feature_data.size
    sparsity = (zero_count / total_elements) * 100
    print(f"  Sparsity: {sparsity:.2f}% (zeros)")
    if name == "Train":
        # Label statistics
        label_counts = df['labels'].str.split().str.len()
        print(f"  Labels per sample - Mean: {label_counts.mean():.2f}, Max: {label_counts.max()}")

get_stats(df_train, "Train")
get_stats(df_test, "Test")

print("[5/5] Data validation complete!")

[5/5] Data quality summary report

Train dataset:
  Rows: 15,377
  Features: 5000
  Feature value range: [0.0000, 45781.1000]
  Feature mean: 0.4182
  Feature std: 5.4026
  Sparsity: 95.24% (zeros)
  Labels per sample - Mean: 5.31, Max: 24

Test dataset:
  Rows: 3,971
  Features: 5000
  Feature value range: [0.0000, 40277.3000]
  Feature mean: 0.4129
  Feature std: 5.0570
  Sparsity: 95.34% (zeros)
[5/5] Data validation complete!


Step 3: Preprocessing (normalization)
Goal: Apply min-max normalization (0-1 scaling) to all features

Categorical encoding was not required because all input feature columns were already numeric in the original data and remained numeric after conversion to CSV. Only the labels field is non-numeric, and it is treated as the target, not as an input feature.

In [6]:

# Compute normalization using MaxAbsScaler (sparse-safe)
print("="*70)
print("Step 3: Preprocessing - MaxAbsScaler normalization (sparse-safe)")
print("="*70)

from scipy.sparse import csr_matrix
from sklearn.preprocessing import MaxAbsScaler

print("[1/4] Converting feature data to sparse matrix and fitting scaler...")
# EurLex TF-IDF features are ~95% zeros — sparse format saves ~20x memory vs dense

X_train_raw = csr_matrix(df_train[feature_cols].values)
X_test_raw  = csr_matrix(df_test[feature_cols].values)

print(f"[1/4] Train sparse shape : {X_train_raw.shape}  nnz={X_train_raw.nnz:,}")
print(f"[1/4] Test  sparse shape : {X_test_raw.shape}  nnz={X_test_raw.nnz:,}")

# Fit MaxAbsScaler on TRAINING data only.
# MaxAbsScaler: X_norm = X / max(|X|) per feature.
# Critically — it never subtracts anything, so zeros stay zero → sparsity preserved.
scaler = MaxAbsScaler()
scaler.fit(X_train_raw)

scale_vals = scaler.scale_
constant_features = (scale_vals == 0)
num_constant = int(constant_features.sum())

if num_constant > 0:
    print(f"[1/4] Found {num_constant} zero-only features (scale=0) — kept unchanged")

print(f"[1/4] Scaler fitted on {len(feature_cols)} features")
print(f"      Sample scales: {feature_cols[0]}: {scale_vals[0]:.4f}  |  {feature_cols[100]}: {scale_vals[100]:.4f}")


Step 3: Preprocessing - MaxAbsScaler normalization (sparse-safe)
[1/4] Converting feature data to sparse matrix and fitting scaler...
[1/4] Train sparse shape : (15377, 5000)  nnz=3,662,025
[1/4] Test  sparse shape : (3971, 5000)  nnz=925,783
[1/4] Scaler fitted on 5000 features
      Sample scales: f0: 351.4250  |  f100: 41.1983


In [7]:

# Apply MaxAbsScaler to sparse matrices
print("[2/4] Applying MaxAbsScaler (X_norm = X / max|X| per feature)...")

X_train_normalized = scaler.transform(X_train_raw)   # CSR sparse, stays sparse
X_test_normalized  = scaler.transform(X_test_raw)    # CSR sparse, stays sparse

# Clip any test values slightly above 1.0 (out-of-range test distribution)
print("[2/4] Clipping test features to max 1.0 ...")
X_test_normalized = X_test_normalized.minimum(1.0)
X_test_normalized.eliminate_zeros()

train_density = X_train_normalized.nnz / (X_train_normalized.shape[0] * X_train_normalized.shape[1]) * 100
sparse_mb  = X_train_normalized.data.nbytes / 1e6
dense_mb   = X_train_normalized.shape[0] * X_train_normalized.shape[1] * 8 / 1e6

print(f"[2/4] Train normalized : shape={X_train_normalized.shape}, density={train_density:.1f}%")
print(f"[2/4] Test  normalized : shape={X_test_normalized.shape}")
print(f"[2/4] Memory (sparse)  : ~{sparse_mb:.1f} MB  vs  ~{dense_mb:.0f} MB dense  ({dense_mb/sparse_mb:.0f}x saving)")


[2/4] Applying MaxAbsScaler (X_norm = X / max|X| per feature)...
[2/4] Clipping test features to max 1.0 ...
[2/4] Train normalized : shape=(15377, 5000), density=4.8%
[2/4] Test  normalized : shape=(3971, 5000)
[2/4] Memory (sparse)  : ~29.3 MB  vs  ~615 MB dense  (21x saving)


In [8]:

# Verify normalization on sparse matrices
print("[3/4] Verifying normalization...")

train_norm_max = float(X_train_normalized.max())
test_norm_max  = float(X_test_normalized.max())
# For sparse matrices the true minimum is always 0 (zeros are implicit, not stored)
train_stored_min = float(X_train_normalized.data.min()) if X_train_normalized.nnz > 0 else 0.0

print(f"[3/4] Train max value          : {train_norm_max:.6f}  (expect ≤ 1.0)")
print(f"[3/4] Train min (stored, >0)   : {train_stored_min:.6f}  (implicit zeros not stored)")
print(f"[3/4] Test  max value          : {test_norm_max:.6f}  (after clipping)")

if train_norm_max > 1.001:
    print("[3/4] !! Warning: train has values > 1.0 — check scaler")
elif test_norm_max > 1.001:
    print("[3/4] !! Warning: test has values > 1.0 after clipping — investigate")
else:
    print("[3/4] Normalization verified OK — all values in [0, 1]")


[3/4] Verifying normalization...
[3/4] Train max value          : 1.000000  (expect ≤ 1.0)
[3/4] Train min (stored, >0)   : 0.000035  (implicit zeros not stored)
[3/4] Test  max value          : 1.000000  (after clipping)
[3/4] Normalization verified OK — all values in [0, 1]


In [9]:

# Save normalized feature matrices as sparse .npz  +  metadata as CSV
print("[4/4] Saving normalized datasets...")

from scipy.sparse import save_npz

OUT_TRAIN_NPZ  = Path("X_train_normalized.npz")
OUT_TEST_NPZ   = Path("X_test_normalized.npz")
OUT_TRAIN_META = Path("train_meta.csv")
OUT_TEST_META  = Path("test_meta.csv")

# Sparse feature matrices  (~37 MB vs ~738 MB dense CSV)
save_npz(OUT_TRAIN_NPZ, X_train_normalized.tocsr())
save_npz(OUT_TEST_NPZ,  X_test_normalized.tocsr())

# Row metadata (row_id + labels) — kept as small CSV since it has no features
df_train[['row_id', 'labels']].to_csv(OUT_TRAIN_META, index=False)
df_test[['row_id']].to_csv(OUT_TEST_META, index=False)

print(f"[4/4] Saved {OUT_TRAIN_NPZ}  ({OUT_TRAIN_NPZ.stat().st_size/1e6:.1f} MB)")
print(f"[4/4] Saved {OUT_TEST_NPZ}")
print(f"[4/4] Saved {OUT_TRAIN_META}  (row_id + labels for all training rows)")
print(f"[4/4] Saved {OUT_TEST_META}   (row_id only)")
print()
print("[4/4] NOTE: eurlex_normalized.csv and eurlex_test_normalized.csv are NO LONGER produced.")
print("      Milestone 2 must load X_train_normalized.npz + train_meta.csv instead.")


[4/4] Saving normalized datasets...
[4/4] Saved X_train_normalized.npz  (23.3 MB)
[4/4] Saved X_test_normalized.npz
[4/4] Saved train_meta.csv  (row_id + labels for all training rows)
[4/4] Saved test_meta.csv   (row_id only)

[4/4] NOTE: eurlex_normalized.csv and eurlex_test_normalized.csv are NO LONGER produced.
      Milestone 2 must load X_train_normalized.npz + train_meta.csv instead.


Testing preprocessing on sample data

In [10]:

# Test preprocessing on sample data
print("="*70)
print("Testing preprocessing on sample data")
print("="*70)

sample_size = 100
X_sample    = X_train_normalized[:sample_size]          # sparse slice
meta_sample = df_train[['row_id', 'labels']].head(sample_size)

print(f"[1/3] Created sparse sample: {X_sample.shape}  nnz={X_sample.nnz}")
print(f"      dtype      : {X_sample.dtype}")
print(f"      max value  : {float(X_sample.max()):.4f}")
print(f"      density    : {X_sample.nnz / (X_sample.shape[0]*X_sample.shape[1])*100:.1f}%")

print(f"\n[2/3] Label format validation:")
print(f"      Sample label (row 0): '{meta_sample['labels'].iloc[0]}'")

def parse_labels(label_str):
    if pd.isna(label_str) or str(label_str).strip() == '':
        return []
    return [int(idx.split(':')[0]) for idx in str(label_str).split()]

sample_labels_parsed = [parse_labels(lbl) for lbl in meta_sample['labels'].values[:5]]
print(f"      Parsed labels (first 5 rows):")
for i, lbl in enumerate(sample_labels_parsed):
    print(f"        Row {i}: {len(lbl)} labels — {lbl[:5]}{'...' if len(lbl) > 5 else ''}")

print("\n[3/3] Sample preprocessing test successful!")
print("Ready for Milestone 2: Binary Classifier Training")


Testing preprocessing on sample data
[1/3] Created sparse sample: (100, 5000)  nnz=25688
      dtype      : float64
      max value  : 1.0000
      density    : 5.1%

[2/3] Label format validation:
      Sample label (row 0): '832:1 1070:1 1337:1 1626:1 2971:1 3801:1'
      Parsed labels (first 5 rows):
        Row 0: 6 labels — [832, 1070, 1337, 1626, 2971]...
        Row 1: 4 labels — [855, 1848, 2701, 3083]
        Row 2: 6 labels — [207, 1109, 1419, 2079, 3102]...
        Row 3: 4 labels — [238, 637, 1358, 2742]
        Row 4: 6 labels — [710, 1623, 2325, 2437, 2564]...

[3/3] Sample preprocessing test successful!
Ready for Milestone 2: Binary Classifier Training


Save normalization parameters

In [11]:

# Save fitted normalization scaler for future use
print("="*70)
print("Saving normalization scaler (MaxAbsScaler)")
print("="*70)

import pickle

norm_params = {
    'scaler'       : scaler,         # MaxAbsScaler fitted on training data
    'feature_cols' : feature_cols,   # list of 5000 feature column names
    'num_constant' : num_constant,   # number of all-zero features
}

NORM_PARAMS_FILE = Path("normalization_params.pkl")
with open(NORM_PARAMS_FILE, 'wb') as f:
    pickle.dump(norm_params, f)

print(f"Saved: {NORM_PARAMS_FILE}")
print(f"  Scaler type : MaxAbsScaler (sparse-safe, preserves zero entries)")
print(f"  Features    : {len(feature_cols)}")
print(f"  Zero-only   : {num_constant}")
print()
print("To normalize new data:")
print("  import pickle")
print("  from scipy.sparse import csr_matrix")
print("  params = pickle.load(open('normalization_params.pkl','rb'))")
print("  X_new_sparse = csr_matrix(X_new_dense)")
print("  X_new_normalized = params['scaler'].transform(X_new_sparse)")


Saving normalization scaler (MaxAbsScaler)
Saved: normalization_params.pkl
  Scaler type : MaxAbsScaler (sparse-safe, preserves zero entries)
  Features    : 5000
  Zero-only   : 0

To normalize new data:
  import pickle
  from scipy.sparse import csr_matrix
  params = pickle.load(open('normalization_params.pkl','rb'))
  X_new_sparse = csr_matrix(X_new_dense)
  X_new_normalized = params['scaler'].transform(X_new_sparse)


Categorical encoding: The original dataset was in .mat format where features were already stored as numeric matrices. Upon conversion to CSV, all 5000 feature columns retained their numeric form. No further categorical encoding was required.

Now implementing vertical data partioning/sharding

Since the dataset is "very wide" (5000 features), the bottleneck is feature dimensionality, not considering row count for now.
Each node receives all rows but a subset of feature columns, plus row_id and labels.
This allows each node to independently train binary classifiers on its feature slice.

In [12]:

# Partitioning setup — uses the already-normalized sparse matrix (no CSV re-load needed)
print("="*70)
print("Step 4: Vertical Data Partitioning (Column-Based Sharding — Sparse)")
print("="*70)

import math
import os
import pickle
from pathlib import Path

# X_train_normalized is already in memory as a sparse CSR matrix from Step 3
X_to_partition = X_train_normalized   # shape: (n_rows, 5000)

NUM_NODES         = 4
total_features    = X_to_partition.shape[1]
total_rows        = X_to_partition.shape[0]
features_per_node = math.ceil(total_features / NUM_NODES)

print(f"[1/4] Partitioning strategy : Vertical column-based sharding (sparse .npz)")
print(f"[1/4] Total rows            : {total_rows:,}")
print(f"[1/4] Total feature columns : {total_features:,}")
print(f"[1/4] Number of nodes       : {NUM_NODES}")
print(f"[1/4] Features per node (≈) : {features_per_node:,}")
print(f"[1/4] Row metadata          : stored separately in train_meta.csv")


Step 4: Vertical Data Partitioning (Column-Based Sharding — Sparse)
[1/4] Partitioning strategy : Vertical column-based sharding (sparse .npz)
[1/4] Total rows            : 15,377
[1/4] Total feature columns : 5,000
[1/4] Number of nodes       : 4
[1/4] Features per node (≈) : 1,250
[1/4] Row metadata          : stored separately in train_meta.csv


In [13]:

# Implement vertical sharding on sparse matrix
print("[2/4] Implementing vertical sharding on sparse matrix...")

from scipy.sparse import save_npz

PARTITION_DIR = Path("partitions")
PARTITION_DIR.mkdir(exist_ok=True)

def vertical_shard_sparse(X_sparse, feature_cols, num_nodes, output_dir):
    """
    Vertically partition a sparse CSR matrix into column-based shards.
    Each shard: all rows, a contiguous subset of feature columns.
    Saved as .npz (sparse, compact). Row metadata lives in train_meta.csv.
    """
    partitions = []
    features_per_shard = math.ceil(X_sparse.shape[1] / num_nodes)

    for node_id in range(num_nodes):
        start_col = node_id * features_per_shard
        end_col   = min(start_col + features_per_shard, X_sparse.shape[1])

        shard = X_sparse[:, start_col:end_col].tocsr()
        shard_path = output_dir / f"partition_{node_id}.npz"
        save_npz(shard_path, shard)

        feat_start = feature_cols[start_col]
        feat_end   = feature_cols[end_col - 1]

        partitions.append({
            'node_id'       : node_id,
            'start_col_idx' : start_col,
            'end_col_idx'   : end_col - 1,
            'num_features'  : shard.shape[1],
            'num_rows'      : shard.shape[0],
            'feature_range' : (feat_start, feat_end),
            'path'          : str(shard_path),
        })

        print(f"[2/4] Node {node_id}: cols {start_col}–{end_col-1} "
              f"({shard.shape[1]:,} features, {shard.shape[0]:,} rows, "
              f"nnz={shard.nnz:,}) → {shard_path}")

    return partitions

partitions = vertical_shard_sparse(X_to_partition, feature_cols, NUM_NODES, PARTITION_DIR)


[2/4] Implementing vertical sharding on sparse matrix...
[2/4] Node 0: cols 0–1249 (1,250 features, 15,377 rows, nnz=966,820) → partitions\partition_0.npz
[2/4] Node 1: cols 1250–2499 (1,250 features, 15,377 rows, nnz=869,637) → partitions\partition_1.npz
[2/4] Node 2: cols 2500–3749 (1,250 features, 15,377 rows, nnz=909,552) → partitions\partition_2.npz
[2/4] Node 3: cols 3750–4999 (1,250 features, 15,377 rows, nnz=916,016) → partitions\partition_3.npz


In [14]:
# Cell 14: Verify workload balance across nodes
print("[3/4] Verifying workload balance across nodes...")

feat_counts = [p['num_features'] for p in partitions]
max_feat = max(feat_counts)
min_feat = min(feat_counts)
imbalance = ((max_feat - min_feat) / max_feat) * 100

print(f"\n[3/4] Feature distribution across nodes:")
for p in partitions:
    bar = "█" * (p['num_features'] // 50)
    print(f"       Node {p['node_id']}: {p['num_features']:,} features  "
          f"({p['feature_range'][0]} → {p['feature_range'][1]})  {bar}")

print(f"\n[3/4] Max features on any node : {max_feat:,}")
print(f"[3/4] Min features on any node : {min_feat:,}")
print(f"[3/4] Feature imbalance        : {imbalance:.2f}%")

if imbalance < 5:
    print("[3/4] ✓ Workload is well balanced across nodes")
else:
    print("[3/4] ⚠ Slight imbalance (last node gets fewer features if not divisible) — acceptable")

# Also confirm every node has the same number of rows
row_counts = [p['num_rows'] for p in partitions]
assert len(set(row_counts)) == 1, "Row count mismatch across partitions!"
print(f"[3/4] ✓ All nodes hold identical row count: {row_counts[0]:,}")

[3/4] Verifying workload balance across nodes...

[3/4] Feature distribution across nodes:
       Node 0: 1,250 features  (f0 → f1249)  █████████████████████████
       Node 1: 1,250 features  (f1250 → f2499)  █████████████████████████
       Node 2: 1,250 features  (f2500 → f3749)  █████████████████████████
       Node 3: 1,250 features  (f3750 → f4999)  █████████████████████████

[3/4] Max features on any node : 1,250
[3/4] Min features on any node : 1,250
[3/4] Feature imbalance        : 0.00%
[3/4] ✓ Workload is well balanced across nodes
[3/4] ✓ All nodes hold identical row count: 15,377


In [15]:

# Verify cross-node communication overhead (sparse .npz partitions)
print("[4/4] Verifying cross-node communication overhead...")

from scipy.sparse import load_npz

total_features_seen = 0
shard_rows = []

for p in partitions:
    shard = load_npz(p['path'])
    total_features_seen += shard.shape[1]
    shard_rows.append(shard.shape[0])

# Check 1: total features = original (no overlap, no loss — column ranges are disjoint)
if total_features_seen == X_to_partition.shape[1]:
    print("[4/4] ✓ Feature columns cleanly divided — zero redundancy across nodes")
    print(f"[4/4]   Total features across shards: {total_features_seen:,}")
else:
    print(f"[4/4] !! Mismatch: expected {X_to_partition.shape[1]}, got {total_features_seen}")

# Check 2: every shard has the same number of rows
if len(set(shard_rows)) == 1:
    print(f"[4/4] ✓ All nodes hold identical row count: {shard_rows[0]:,}")
else:
    print(f"[4/4] !! Row count mismatch: {shard_rows}")

print(f"\n[4/4] Partition summary:")
print(f"       Total features distributed : {total_features_seen:,}")
print(f"       Rows on every node         : {shard_rows[0]:,}")
print(f"       Format                     : Sparse .npz (zero-preserving, compact)")
print(f"       Cross-node redundancy      : NONE (disjoint column ranges)")
print(f"       Files: {[f'partition_{i}.npz' for i in range(NUM_NODES)]}")

# Save partition metadata (paths now point to .npz files)
partition_meta = {
    'num_nodes'   : NUM_NODES,
    'feature_cols': feature_cols,
    'partitions'  : [
        {
            'node_id'      : p['node_id'],
            'path'         : p['path'],
            'num_features' : p['num_features'],
            'feature_range': p['feature_range'],
            'start_col_idx': p['start_col_idx'],
            'end_col_idx'  : p['end_col_idx'],
        }
        for p in partitions
    ]
}
with open("partition_metadata.pkl", "wb") as f:
    pickle.dump(partition_meta, f)

print(f"\n[4/4] Partition metadata saved to partition_metadata.pkl")


[4/4] Verifying cross-node communication overhead...
[4/4] ✓ Feature columns cleanly divided — zero redundancy across nodes
[4/4]   Total features across shards: 5,000
[4/4] ✓ All nodes hold identical row count: 15,377

[4/4] Partition summary:
       Total features distributed : 5,000
       Rows on every node         : 15,377
       Format                     : Sparse .npz (zero-preserving, compact)
       Cross-node redundancy      : NONE (disjoint column ranges)
       Files: ['partition_0.npz', 'partition_1.npz', 'partition_2.npz', 'partition_3.npz']

[4/4] Partition metadata saved to partition_metadata.pkl


Now setting up the distributed environment for deploying the tasks to simulated worker nodes

In [ ]:

# Setting up LocalCluster
print("="*70)
print("Step 5: Distributed Framework Setup (Dask)")
print("="*70)

import pickle
import time
import pandas as pd
import numpy as np
from pathlib import Path

try:
    import dask
    import dask.dataframe as dd
    from dask.distributed import Client, LocalCluster, wait, as_completed
    print(f"[1/4] Dask version : {dask.__version__}")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "dask[distributed]", "-q"])
    import dask
    import dask.dataframe as dd
    from dask.distributed import Client, LocalCluster, wait, as_completed
    print(f"[1/4] Dask installed and imported. Version: {dask.__version__}")

# Load partition metadata produced by Fatima
with open("partition_metadata.pkl", "rb") as f:
    partition_meta = pickle.load(f)

NUM_NODES  = partition_meta["num_nodes"]        # 4
PARTITIONS = partition_meta["partitions"]        # list of dicts (paths are .npz)

print(f"[1/4] Loaded partition metadata")
print(f"      Nodes              : {NUM_NODES}")
print(f"      Partition files    : {[p['path'] for p in PARTITIONS]}")

# Spin up a LocalCluster with one worker per partition (simulates 4 nodes)
print("[1/4] Starting Dask LocalCluster with 4 workers...")
cluster = LocalCluster(
    n_workers=NUM_NODES,
    threads_per_worker=1,   # 1 thread per worker = true parallelism
    memory_limit="4GB",
    silence_logs=True
)
client = Client(cluster)

print(f"[1/4] Cluster dashboard : {client.dashboard_link}")
print(f"[1/4] Workers registered : {len(client.scheduler_info()['workers'])}")
print("[1/4] Dask distributed framework ready.")


Step 5: Distributed Framework Setup (Dask)


2026-04-22 15:03:08,282 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-04-22 15:03:08,300 - distributed.scheduler - INFO - State start
2026-04-22 15:03:08,344 - distributed.scheduler - INFO -   Scheduler at:     tcp://127.0.0.1:52697
2026-04-22 15:03:08,344 - distributed.scheduler - INFO -   dashboard at:  http://127.0.0.1:8787/status
2026-04-22 15:03:08,344 - distributed.scheduler - INFO - Registering Worker plugin shuffle


[1/4] Dask version : 2026.3.0
[1/4] Loaded partition metadata
      Nodes              : 4
      Partition files    : ['partitions\\partition_0.npz', 'partitions\\partition_1.npz', 'partitions\\partition_2.npz', 'partitions\\partition_3.npz']
[1/4] Starting Dask LocalCluster with 4 workers...


2026-04-22 15:03:08,498 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:52703'
2026-04-22 15:03:08,511 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:52701'
2026-04-22 15:03:08,511 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:52700'
2026-04-22 15:03:08,535 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:52702'
2026-04-22 15:03:10,675 - distributed.scheduler - INFO - Register worker addr: tcp://127.0.0.1:52087 name: 3
2026-04-22 15:03:10,777 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:52087
2026-04-22 15:03:10,777 - distributed.core - INFO - Starting established connection to tcp://127.0.0.1:52092
2026-04-22 15:03:10,782 - distributed.scheduler - INFO - Register worker addr: tcp://127.0.0.1:52096 name: 0
2026-04-22 15:03:10,786 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:52096
2026-04-22 15:03:10,788 - distributed.core - IN

[1/4] Cluster dashboard : http://127.0.0.1:8787/status
[1/4] Workers registered : 4
[1/4] Dask distributed framework ready.


2026-04-22 15:03:12,042 - distributed.scheduler - INFO - Remove client Client-780430e5-3e32-11f1-8384-c54db532ff72
2026-04-22 15:03:12,042 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:52099; closing.
2026-04-22 15:03:12,047 - distributed.scheduler - INFO - Remove client Client-780430e5-3e32-11f1-8384-c54db532ff72
2026-04-22 15:03:12,057 - distributed.scheduler - INFO - Close client connection: Client-780430e5-3e32-11f1-8384-c54db532ff72
2026-04-22 15:03:12,078 - distributed.scheduler - INFO - Retire worker addresses (stimulus_id='retire-workers-1776852192.0757887') (0, 1, 2, 3)
2026-04-22 15:03:12,125 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:52098; closing.
2026-04-22 15:03:12,130 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:52091; closing.
2026-04-22 15:03:12,136 - distributed.core - INFO - Received 'close-stream' from tcp://127.0.0.1:52095; closing.
2026-04-22 15:03:12,273 - distributed.scheduler 

In [17]:

# Defining worker-side preprocessing task and deploying to nodes
print("="*70)
print("[2/4] Deploying preprocessing tasks to worker nodes")
print("="*70)

# Each worker receives its partition .npz path and validates it independently.
# No data is shared between workers.

def worker_preprocess_task(partition_path: str) -> dict:
    """
    Worker-side preprocessing task.
    Loads a sparse .npz partition, validates range and sparsity, returns summary.
    """
    import scipy.sparse as sp
    import numpy as np
    import os
    import time

    start = time.time()
    node_id = int(partition_path.split("partition_")[1].replace(".npz", ""))

    # Load sparse partition
    shard = sp.load_npz(partition_path)

    # Validate: all stored values in [0, 1]  (zeros are implicit — always in range)
    feat_max = float(shard.max()) if shard.nnz > 0 else 0.0
    feat_min = float(shard.min()) if shard.nnz > 0 else 0.0
    in_range = (feat_min >= 0.0) and (feat_max <= 1.001)

    # Sparsity
    sparsity_pct = round((1 - shard.nnz / (shard.shape[0] * shard.shape[1])) * 100, 2)

    elapsed = time.time() - start
    return {
        "node_id"     : node_id,
        "pid"         : os.getpid(),
        "rows"        : shard.shape[0],
        "features"    : shard.shape[1],
        "feat_min"    : round(feat_min, 6),
        "feat_max"    : round(feat_max, 6),
        "in_range"    : in_range,
        "missing"     : 0,         # sparse .npz has no NaN by construction
        "sparsity_pct": sparsity_pct,
        "elapsed_sec" : round(elapsed, 3),
    }

# Submit one task per partition
futures = []
for p in PARTITIONS:
    future = client.submit(
        worker_preprocess_task,
        p["path"],
        pure=False
    )
    futures.append(future)
    print(f"  -> Dispatched task for Node {p['node_id']} ({p['path']})")

print(f"[2/4] {len(futures)} tasks dispatched to Dask workers.")
print("[2/4] Tasks are executing in parallel across workers...")


[2/4] Deploying preprocessing tasks to worker nodes
  -> Dispatched task for Node 0 (partitions\partition_0.npz)
  -> Dispatched task for Node 1 (partitions\partition_1.npz)
  -> Dispatched task for Node 2 (partitions\partition_2.npz)
  -> Dispatched task for Node 3 (partitions\partition_3.npz)
[2/4] 4 tasks dispatched to Dask workers.
[2/4] Tasks are executing in parallel across workers...


In [18]:
#Monitoring task execution and collect results

print("="*70)
print("[3/4] Monitoring distributed task execution")
print("="*70)

results = []
dispatch_time = time.time()

# as_completed yields futures in the order they finish
print("Node | PID    | Rows   | Features | In-Range | Missing | Sparsity | Time(s)")
print("  " + "-"*75)

for future in as_completed(futures):
    r = future.result()
    results.append(r)
    status = "OK" if (r["in_range"] and r["missing"] == 0) else "WARN"
    print(f"  {r['node_id']}    | {r['pid']}  | "
          f"{r['rows']:,}  | {r['features']:,}     | "
          f"{str(r['in_range']):5}    | {r['missing']:,}      | "
          f"{r['sparsity_pct']}%  | {r['elapsed_sec']}s  [{status}]")

total_wall_time = round(time.time() - dispatch_time, 3)
results.sort(key=lambda x: x["node_id"])   # restore node order

print(f"[3/4] All {len(results)} worker tasks completed.")
print(f"[3/4] Total wall-clock time (parallel): {total_wall_time}s")

# Aggregate validation checks
all_in_range = all(r["in_range"] for r in results)
total_missing = sum(r["missing"] for r in results)
total_rows_processed = sum(r["rows"] for r in results)
total_features_processed = sum(r["features"] for r in results)

print(f"[3/4] Aggregate validation:")
print(f"      All partitions in [0,1] range : {all_in_range}")
print(f"      Total missing values           : {total_missing}")
print(f"      Total rows processed           : {total_rows_processed:,} "
      f"(= {len(results)} nodes × {results[0]['rows']:,} rows)")
print(f"      Total features processed       : {total_features_processed:,} "
      f"(= {len(results)} nodes × {results[0]['features']:,} features/node)")

[3/4] Monitoring distributed task execution
Node | PID    | Rows   | Features | In-Range | Missing | Sparsity | Time(s)
  ---------------------------------------------------------------------------
  2    | 13964  | 15,377  | 1,250     | True     | 0      | 95.27%  | 0.114s  [OK]
  0    | 6976  | 15,377  | 1,250     | True     | 0      | 94.97%  | 0.114s  [OK]
  1    | 28480  | 15,377  | 1,250     | True     | 0      | 95.48%  | 0.067s  [OK]
  3    | 9572  | 15,377  | 1,250     | True     | 0      | 95.23%  | 0.116s  [OK]
[3/4] All 4 worker tasks completed.
[3/4] Total wall-clock time (parallel): 0.586s
[3/4] Aggregate validation:
      All partitions in [0,1] range : True
      Total missing values           : 0
      Total rows processed           : 61,508 (= 4 nodes × 15,377 rows)
      Total features processed       : 5,000 (= 4 nodes × 1,250 features/node)


In [19]:

# Distributed system configuration document + shutdown

print("="*70)
print("[4/4] Distributed System Configuration Summary")
print("="*70)

import json

# Collect live scheduler info
scheduler_info = client.scheduler_info()
worker_info    = scheduler_info["workers"]

config_doc = {
    "framework"         : "Dask Distributed",
    "dask_version"      : dask.__version__,
    "cluster_type"      : "LocalCluster (simulated 4-node cluster)",
    "num_workers"       : len(worker_info),
    "threads_per_worker": 1,
    "memory_per_worker" : "4 GB",
    "data_format"       : "Sparse CSR .npz (MaxAbsScaler, sparsity-preserving)",
    "partitioning"      : {
        "strategy"          : "Vertical (column-based) sharding — sparse",
        "num_partitions"    : NUM_NODES,
        "features_per_node" : results[0]["features"],
        "rows_per_node"     : results[0]["rows"],
        "cross_node_comm"   : "None (disjoint column ranges, all rows on every node)"
    },
    "preprocessing_deployed": [
        {
            "node_id"      : r["node_id"],
            "pid"          : r["pid"],
            "status"       : "OK" if (r["in_range"] and r["missing"] == 0) else "WARN",
            "elapsed_sec"  : r["elapsed_sec"],
            "sparsity_pct" : r["sparsity_pct"]
        }
        for r in results
    ],
    "total_wall_time_sec" : total_wall_time,
    "validation" : {
        "all_in_range"  : all_in_range,
        "total_missing" : total_missing
    }
}

# Print formatted configuration
print(json.dumps(config_doc, indent=4))

# Save to disk for reproducibility
CONFIG_FILE = Path("distributed_config.json")
with open(CONFIG_FILE, "w") as f:
    json.dump(config_doc, f, indent=4)
print(f"[4/4] Configuration saved to: {CONFIG_FILE}")

# Graceful shutdown of the cluster
client.close()
cluster.close()
print("[4/4] Dask cluster shut down cleanly.")
print("="*70)
print("Milestone 1 COMPLETE")
print("  Mustafa  -> Data ingestion & validation     [DONE]")
print("  Sohaib   -> MaxAbsScaler normalisation      [DONE]")
print("  Fatima   -> Vertical partitioning (sparse)  [DONE]")
print("  Sameem   -> Dask framework & deployment     [DONE]")
print("="*70)
print("Output artefacts:")
print("  X_train_normalized.npz         (Sohaib — sparse feature matrix, ~37 MB)")
print("  X_test_normalized.npz          (Sohaib — sparse feature matrix)")
print("  train_meta.csv                 (Sohaib — row_id + labels)")
print("  test_meta.csv                  (Sohaib — row_id only)")
print("  normalization_params.pkl       (Sohaib — MaxAbsScaler)")
print("  partitions/partition_0..3.npz  (Fatima — sparse column shards)")
print("  partition_metadata.pkl         (Fatima)")
print("  distributed_config.json        (Sameem)")
print()
print("OBSOLETE files (no longer produced — do NOT use):")
print("  eurlex_normalized.csv          <- replaced by X_train_normalized.npz + train_meta.csv")
print("  eurlex_test_normalized.csv     <- replaced by X_test_normalized.npz + test_meta.csv")
print("  partitions/partition_0..3.csv  <- replaced by partition_0..3.npz")
print("Ready for Milestone 2: Parallel Binary Classifier Training.")


2026-04-22 15:03:12,085 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:52700'. Reason: nanny-close
2026-04-22 15:03:12,089 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-22 15:03:12,091 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:52701'. Reason: nanny-close
2026-04-22 15:03:12,095 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-22 15:03:12,098 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:52702'. Reason: nanny-close
2026-04-22 15:03:12,104 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-22 15:03:12,107 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:52703'. Reason: nanny-close
2026-04-22 15:03:12,110 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close


[4/4] Distributed System Configuration Summary
{
    "framework": "Dask Distributed",
    "dask_version": "2026.3.0",
    "cluster_type": "LocalCluster (simulated 4-node cluster)",
    "num_workers": 4,
    "threads_per_worker": 1,
    "memory_per_worker": "4 GB",
    "data_format": "Sparse CSR .npz (MaxAbsScaler, sparsity-preserving)",
    "partitioning": {
        "strategy": "Vertical (column-based) sharding \u2014 sparse",
        "num_partitions": 4,
        "features_per_node": 1250,
        "rows_per_node": 15377,
        "cross_node_comm": "None (disjoint column ranges, all rows on every node)"
    },
    "preprocessing_deployed": [
        {
            "node_id": 0,
            "pid": 6976,
            "status": "OK",
            "elapsed_sec": 0.114,
            "sparsity_pct": 94.97
        },
        {
            "node_id": 1,
            "pid": 28480,
            "status": "OK",
            "elapsed_sec": 0.067,
            "sparsity_pct": 95.48
        },
        {
    

2026-04-22 15:03:14,695 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:52700' closed.
2026-04-22 15:03:14,701 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:52701' closed.
2026-04-22 15:03:14,705 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:52702' closed.
2026-04-22 15:03:14,705 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:52703' closed.


[4/4] Dask cluster shut down cleanly.
Milestone 1 COMPLETE
  Mustafa  -> Data ingestion & validation     [DONE]
  Sohaib   -> MaxAbsScaler normalisation      [DONE]
  Fatima   -> Vertical partitioning (sparse)  [DONE]
  Sameem   -> Dask framework & deployment     [DONE]
Output artefacts:
  X_train_normalized.npz         (Sohaib — sparse feature matrix, ~37 MB)
  X_test_normalized.npz          (Sohaib — sparse feature matrix)
  train_meta.csv                 (Sohaib — row_id + labels)
  test_meta.csv                  (Sohaib — row_id only)
  normalization_params.pkl       (Sohaib — MaxAbsScaler)
  partitions/partition_0..3.npz  (Fatima — sparse column shards)
  partition_metadata.pkl         (Fatima)
  distributed_config.json        (Sameem)

OBSOLETE files (no longer produced — do NOT use):
  eurlex_normalized.csv          <- replaced by X_train_normalized.npz + train_meta.csv
  eurlex_test_normalized.csv     <- replaced by X_test_normalized.npz + test_meta.csv
  partitions/partition_0

## Analytics: Partition Balance Analysis
Measures whether the vertical column shards produced in Step 4 are evenly distributed across workers in terms of feature count, non-zero elements, and file size. Saved as `partition_balance.png`.

In [ ]:
# ── ANALYTICS: Partition Balance Analysis ─────────────────────────────────────
# Added for PDC report: measures whether vertical column shards are evenly sized.
# No existing logic is changed — this cell only reads already-generated .npz files.

import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.sparse import load_npz
from pathlib import Path

print("=" * 60)
print("Partition Balance Analysis")
print("=" * 60)

PARTITION_DIR = Path("partitions")
partition_files = sorted(PARTITION_DIR.glob("partition_*.npz"))

if not partition_files:
    print("No partition files found. Run partitioning cells first.")
else:
    labels_x    = []
    nnz_vals    = []
    size_mb     = []
    feat_counts = []
    row_counts  = []

    for pf in partition_files:
        shard = load_npz(pf)
        pid   = pf.stem  # e.g. "partition_0"
        labels_x.append(pid.replace("partition_", "Worker "))
        nnz_vals.append(shard.nnz)
        size_mb.append(os.path.getsize(pf) / 1e6)
        feat_counts.append(shard.shape[1])
        row_counts.append(shard.shape[0])
        print(f"  {pid}: shape={shard.shape}  nnz={shard.nnz:,}  size={os.path.getsize(pf)/1e6:.2f} MB")

    # Imbalance metrics
    nnz_max, nnz_min = max(nnz_vals), min(nnz_vals)
    nnz_imbalance    = (nnz_max - nnz_min) / nnz_max * 100
    feat_max, feat_min = max(feat_counts), min(feat_counts)
    feat_imbalance     = (feat_max - feat_min) / feat_max * 100

    print(f"\nFeature imbalance across partitions : {feat_imbalance:.2f}%")
    print(f"NNZ     imbalance across partitions : {nnz_imbalance:.2f}%")

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1 — Features per partition
    bars = axes[0].bar(labels_x, feat_counts, color='steelblue', edgecolor='white')
    axes[0].axhline(sum(feat_counts)/len(feat_counts), color='red', linestyle='--', label='Mean')
    axes[0].set_title("Feature Columns per Partition")
    axes[0].set_xlabel("Partition (Worker)")
    axes[0].set_ylabel("Number of Features")
    axes[0].legend()
    for bar, v in zip(bars, feat_counts):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(v),
                     ha='center', va='bottom', fontsize=9)

    # Panel 2 — Non-zeros per partition
    bars2 = axes[1].bar(labels_x, nnz_vals, color='darkorange', edgecolor='white')
    axes[1].axhline(sum(nnz_vals)/len(nnz_vals), color='red', linestyle='--', label='Mean')
    axes[1].set_title("Non-Zero Elements per Partition")
    axes[1].set_xlabel("Partition (Worker)")
    axes[1].set_ylabel("Non-Zero Count")
    axes[1].legend()
    for bar, v in zip(bars2, nnz_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(nnz_vals)*0.01,
                     f'{v:,}', ha='center', va='bottom', fontsize=8)

    # Panel 3 — File size per partition
    bars3 = axes[2].bar(labels_x, size_mb, color='seagreen', edgecolor='white')
    axes[2].axhline(sum(size_mb)/len(size_mb), color='red', linestyle='--', label='Mean')
    axes[2].set_title("File Size per Partition (MB)")
    axes[2].set_xlabel("Partition (Worker)")
    axes[2].set_ylabel("Size (MB)")
    axes[2].legend()
    for bar, v in zip(bars3, size_mb):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(size_mb)*0.01,
                     f'{v:.2f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle("Vertical Partition Balance Analysis", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("partition_balance.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: partition_balance.png")
